# Importations alimentaires des Comores

Notebook du projet FAOSTAT. Les données brutes sont conservées sans modification.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
FIGURES_DIR = ROOT_DIR / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement et contrôle

In [6]:
RAW_FILE = RAW_DIR / "FAOSTAT_data.csv"
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Place le fichier FAOSTAT_data.csv dans {RAW_DIR}")
raw = pd.read_csv(RAW_FILE, low_memory=False)
print(raw.shape)
display(raw.head())
display(raw.isna().sum().to_frame("valeurs_manquantes"))

(17097, 16)


,Domain Code,Domain,Reporter Country Code (M49),Reporter Countries,Partner Country Code (M49),Partner Countries,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description
0,TM,Detailed trade matrix,174,Comoros,4,Afghanistan,5610,Import quantity,23995.03,Flour of mustard seed,2019,2019,t,0.70,A,Official figure
1,TM,Detailed trade matrix,174,Comoros,4,Afghanistan,5622,Import value,23995.03,Flour of mustard seed,2019,2019,1000 USD,0.00,A,Official figure
2,TM,Detailed trade matrix,174,Comoros,4,Afghanistan,5610,Import quantity,F1232,Food preparations n.e.c.,2021,2021,t,1.00,A,Official figure
3,TM,Detailed trade matrix,174,Comoros,4,Afghanistan,5622,Import value,F1232,Food preparations n.e.c.,2021,2021,1000 USD,0.00,A,Official figure
4,TM,Detailed trade matrix,174,Comoros,4,Afghanistan,5610,Import quantity,2351f,Raw cane or beet sugar (centrifugal only),2017,2017,t,25.48,A,Official figure


,valeurs_manquantes
Domain Code,0
Domain,0
Reporter Country Code (M49),0
Reporter Countries,0
Partner Country Code (M49),0
Partner Countries,0
Element Code,0
Element,0
Item Code (CPC),0
Item,0


## 2. Filtrage du périmètre

On retient les importations des Comores : quantités en tonnes et valeurs en milliers USD. Les unités de comptage ne sont pas converties en tonnes.

In [7]:
imports = raw.loc[
    (raw["Reporter Countries"] == "Comoros")
    & (
        ((raw["Element"] == "Import quantity") & (raw["Unit"] == "t"))
        | ((raw["Element"] == "Import value") & (raw["Unit"] == "1000 USD"))
    )
].copy()
imports["Year"] = pd.to_numeric(imports["Year"], errors="coerce")
imports["Value"] = pd.to_numeric(imports["Value"], errors="coerce")
display(imports.groupby(["Element", "Unit"]).size().rename("observations").to_frame())

,,observations
Element,Unit,
Import quantity,t,8079
Import value,1000 USD,8162


## 3. Couverture temporelle et qualité

In [8]:
years = sorted(imports["Year"].dropna().astype(int).unique())
missing_years = sorted(set(range(min(years), max(years)+1)) - set(years)) if years else []
print("Années disponibles :", years)
print("Années absentes :", missing_years)

display(imports.groupby(["Flag", "Flag Description"], dropna=False).size()
        .reset_index(name="observations").sort_values("observations", ascending=False))

Années disponibles : [np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Années absentes : [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]


,Flag,Flag Description,observations
0,A,Official figure,15894
1,E,Estimated value,334
3,X,Figure from international organizations,12
2,I,Imputed value,1


## 4. Catalogue CPC

In [9]:
products = (imports[["Item Code (CPC)", "Item"]].drop_duplicates()
            .sort_values("Item Code (CPC)").reset_index(drop=True))
print("Produits distincts :", len(products))
display(products)

Produits distincts : 344


,Item Code (CPC),Item
0,0111,Wheat
1,0112,Maize (corn)
2,0113,Rice
3,0114,Sorghum
4,0116,Rye
...,...,...
339,F1172,Meat prepared n.e.c.
340,F1232,Food preparations n.e.c.
341,F1243,Fat preparations n.e.c.
342,F1275,hydrogenated oils and fats


## 5. Table de classification

Le notebook crée un modèle à compléter. Les catégories proposées sont `Alimentation humaine`, `Alimentation animale`, `Usage mixte`, `Non alimentaire` et `À vérifier`. Ne classe pas automatiquement un produit à partir de son seul nom.

In [10]:
CLASS_FILE = PROCESSED_DIR / "product_classification.csv"
if CLASS_FILE.exists():
    categories = pd.read_csv(CLASS_FILE, dtype={"Item Code (CPC)": "string"})
else:
    categories = products.copy()
    categories["Product_Category"] = "À vérifier"
    categories["Classification_Comment"] = ""
    categories.to_csv(CLASS_FILE, index=False, encoding="utf-8-sig")
    print("Modèle créé :", CLASS_FILE)

if categories["Item Code (CPC)"].duplicated().any():
    raise ValueError("Codes CPC en double dans la table de classification.")
display(categories["Product_Category"].value_counts(dropna=False))
display(categories.head())

Modèle créé : e:\Projects\importations-alimentaires-comores\data\processed\product_classification.csv


Product_Category
À vérifier    344
Name: count, dtype: int64

,Item Code (CPC),Item,Product_Category,Classification_Comment
0,0111,Wheat,À vérifier,
1,0112,Maize (corn),À vérifier,
2,0113,Rice,À vérifier,
3,0114,Sorghum,À vérifier,
4,0116,Rye,À vérifier,


Compléter et vérifier `data/processed/product_classification.csv` avant d'interpréter les résultats par catégorie.

## 6. Table produit-année-pays

In [11]:
data = imports.pivot_table(
    index=["Year", "Item Code (CPC)", "Item", "Partner Countries"],
    columns="Element", values="Value", aggfunc="sum"
).reset_index()
data.columns.name = None
data = data.rename(columns={
    "Import quantity": "Quantity (t)",
    "Import value": "Value (1000 USD)"
})
data = data.merge(
    categories[["Item Code (CPC)", "Product_Category"]],
    on="Item Code (CPC)", how="left", validate="many_to_one"
)
display(data.head())
print("Produits sans catégorie :", data["Product_Category"].isna().sum())

,Year,Item Code (CPC),Item,Partner Countries,Quantity (t),Value (1000 USD),Product_Category
0,1995,0111,Wheat,France,0.00,0.00,À vérifier
1,1995,0112,Maize (corn),France,0.00,0.00,À vérifier
2,1995,0112,Maize (corn),Madagascar,4.00,2.00,À vérifier
3,1995,0112,Maize (corn),South Africa,32.00,9.00,À vérifier
4,1995,0112,Maize (corn),United Arab Emirates,1.00,2.00,À vérifier


Produits sans catégorie : 0


In [12]:
data.to_csv(PROCESSED_DIR / "imports_comoros_processed.csv",
            index=False, encoding="utf-8-sig")
products.to_csv(PROCESSED_DIR / "product_catalog.csv",
                index=False, encoding="utf-8-sig")
print("Données préparées exportées.")

Données préparées exportées.
